# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print metadata information
metadata = dataset.metadata
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished if hasattr(metadata, 'datePublished') else ''}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We list the available record sets and their fields below.

In [ ]:
# List all record sets (@id and name)
print('Available Record Sets:')
record_sets = dataset.metadata.recordSet if hasattr(dataset.metadata, 'recordSet') else []
for rs in record_sets:
    print(f"  - @id: {getattr(rs, '@id', '')} | Name: {getattr(rs, 'name', '')}")
    if hasattr(rs, 'field'):
        print("    Fields / Columns:")
        for fld in rs.field:
            print(f"      - @id: {getattr(fld, '@id', '')} | Name: {getattr(fld, 'name', '')}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. 
Use the record set and field `@id`s from above. If there are multiple record sets you can extract them all.


In [ ]:
# Gather record set IDs
record_set_ids = []
for rs in record_sets:
    if hasattr(rs, '@id'):
        record_set_ids.append(getattr(rs, '@id'))

print('Record set @ids found:', record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f'First columns in {record_set_id}:', df.columns.tolist())
        display(df.head(3))

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data, or grouping data by key attributes.

*Note: You may need to customize the numeric and group field `@id` below to match your data structure.*

In [ ]:
# Choose a record set for EDA, pick the first if more than one exists
if len(dataframes) == 0:
    print('No tabular data found for EDA.')
else:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f'Record set under analysis: {record_set_id}')
    print('Columns:', df.columns.tolist())

    # Try to detect a numeric field (e.g. floats/integers)
    numeric_field = None
    for col in df.columns:
        try:
            # Try to convert to numeric sample
            pd.to_numeric(df[col], errors='raise')
            numeric_field = col
            break
        except:
            continue
    if numeric_field is None:
        print('No numeric field detected.')
    else:
        print(f'Numeric field automatically selected: {numeric_field}')
        # Filtering
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].mean()
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered rows with {numeric_field} > mean ({threshold:.2f}):\n", filtered_df.head())
        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:\n",
              filtered_df[[numeric_field, f"{numeric_field}_normalized"].copy()].head())

    # Try to find a categorical/group field
    group_field = None
    for col in df.columns:
        if df[col].dtype == object and col != numeric_field:
            group_field = col
            break
    if group_field and numeric_field:
        grouped_df = df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Mean {numeric_field} grouped by {group_field}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

*You can change the numeric or group field if another is more informative in your context!*

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if len(dataframes) > 0 and numeric_field is not None:
    plt.figure(figsize=(8, 4))
    df[numeric_field].hist(bins=30)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Distribution of {numeric_field}')
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10, 5))
        df.boxplot(column=numeric_field, by=group_field, rot=90)
        plt.title(f"{numeric_field} by {group_field}")
        plt.suptitle("")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.tight_layout()
        plt.show()
else:
    print("No suitable numeric field for plotting.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides rich mass spectrometry data on human mast cell extracellular vesicles.
- We successfully loaded the data using the Croissant schema and explored the available structure via record sets and fields (@id references).
- Basic EDA steps including filtering, normalization, and grouping by category were demonstrated on a selected numeric field.
- Visualizations can help detect data distributions or group differences.

**Tip:** For robust analysis, refer to the schema metadata for in-depth field meaning and adjust grouping or selection logic as needed for your research question.